In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

# ============================================================
# PDF-BASED QUESTION ANSWERING CHATBOT USING LANGCHAIN + GEMINI
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL REQUIRED LIBRARIES
# ------------------------------------------------------------

!pip install -q -U google-genai langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu pypdf sentence-transformers


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from google import genai

print("All libraries imported successfully!")


# ------------------------------------------------------------
# 3. SET GEMINI API KEY SECURELY
# ------------------------------------------------------------

# This prevents your API key from appearing in your notebook
# or being uploaded accidentally to GitHub.

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass(
        "Enter your Gemini API Key: "
    )

print("Gemini API key configured successfully!")


# ------------------------------------------------------------
# 4. INITIALIZE GEMINI CLIENT
# ------------------------------------------------------------

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client initialized successfully!")


# ------------------------------------------------------------
# 5. DISPLAY AVAILABLE MODELS
# ------------------------------------------------------------

print("\nChecking available Gemini models...\n")

available_models = []

try:
    for model in client.models.list():
        model_name = getattr(model, "name", None)

        if model_name:
            available_models.append(model_name)

            print(model_name)

except Exception as e:
    print("Could not list models.")
    print("Reason:", e)


# ------------------------------------------------------------
# 6. SELECT GEMINI MODEL
# ------------------------------------------------------------

# Try these models in order.
# The code will automatically select the first available one.

preferred_models = [
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite"
]

MODEL_NAME = None

for preferred_model in preferred_models:

    # Models may be returned as:
    # models/gemini-3.7-flash
    # or gemini-3.7-flash

    if (
        preferred_model in available_models
        or f"models/{preferred_model}" in available_models
    ):
        MODEL_NAME = preferred_model
        break


# If automatic detection fails, use this fallback.
if MODEL_NAME is None:

    MODEL_NAME = "gemini-2.5-flash"

    print(
        "\nAutomatic model detection failed."
    )

    print(
        "Using fallback model:",
        MODEL_NAME
    )

else:

    print(
        "\nSelected Gemini model:",
        MODEL_NAME
    )


# ------------------------------------------------------------
# 7. SPECIFY PDF FILE PATH
# ------------------------------------------------------------

# CHANGE THIS FILE NAME IF YOUR PDF HAS A DIFFERENT NAME

pdf_path = "sem2result.pdf"


# ------------------------------------------------------------
# 8. CHECK WHETHER PDF EXISTS
# ------------------------------------------------------------

if not os.path.exists(pdf_path):

    print("\nERROR: PDF FILE NOT FOUND!")

    print("\nCurrent working directory:")
    print(os.getcwd())

    print("\nFiles available in this folder:")

    for file in os.listdir():
        print(file)

    raise FileNotFoundError(
        f"\nPDF file '{pdf_path}' was not found."
        "\nPlease place the PDF in the same folder "
        "as this Jupyter Notebook."
    )


print("\nPDF found successfully!")
print("PDF Path:", pdf_path)


# ------------------------------------------------------------
# 9. LOAD PDF
# ------------------------------------------------------------

print("\nLoading PDF...")

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("PDF loaded successfully!")

print(
    "Number of pages:",
    len(documents)
)


# ------------------------------------------------------------
# 10. DISPLAY SAMPLE PDF CONTENT
# ------------------------------------------------------------

if len(documents) > 0:

    print("\n")
    print("=" * 70)

    print("SAMPLE CONTENT FROM PDF")

    print("=" * 70)

    print(
        documents[0].page_content[:1000]
    )

    print("\n")
    print("=" * 70)


# ------------------------------------------------------------
# 11. SPLIT PDF INTO TEXT CHUNKS
# ------------------------------------------------------------

print("\nSplitting document into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(
    documents
)

print(
    "Document split successfully!"
)

print(
    "Number of chunks:",
    len(chunks)
)


# ------------------------------------------------------------
# 12. LOAD HUGGINGFACE EMBEDDING MODEL
# ------------------------------------------------------------

print(
    "\nLoading embedding model..."
)

embedding_model = HuggingFaceEmbeddings(
    model_name=
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding model loaded successfully!"
)


# ------------------------------------------------------------
# 13. CREATE FAISS VECTOR DATABASE
# ------------------------------------------------------------

print(
    "\nCreating FAISS vector database..."
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print(
    "FAISS vector database created successfully!"
)


# ------------------------------------------------------------
# 14. CREATE RETRIEVER FUNCTION
# ------------------------------------------------------------

def get_relevant_context(
    question,
    k=4
):

    relevant_documents = (
        vector_store.similarity_search(
            query=question,
            k=k
        )
    )

    context_parts = []

    for i, doc in enumerate(
        relevant_documents
    ):

        page_number = (
            doc.metadata.get(
                "page",
                "Unknown"
            )
        )

        # PyPDFLoader page numbers start from 0
        if isinstance(
            page_number,
            int
        ):
            page_number += 1

        context_parts.append(
            f"""
SOURCE {i + 1}
PAGE: {page_number}

CONTENT:
{doc.page_content}
"""
        )

    context = "\n\n".join(
        context_parts
    )

    return (
        context,
        relevant_documents
    )


print(
    "Retriever function created successfully!"
)


# ------------------------------------------------------------
# 15. CREATE QUESTION-ANSWERING FUNCTION
# ------------------------------------------------------------

def ask_pdf(question):

    # Retrieve relevant PDF chunks
    context, relevant_documents = (
        get_relevant_context(
            question,
            k=4
        )
    )


    # Create RAG prompt
    prompt = f"""
You are a PDF-based Question Answering Chatbot.

Your task is to answer the user's question using ONLY
the information provided in the PDF context below.

IMPORTANT RULES:

1. Answer only from the provided context.
2. Do not use outside knowledge.
3. Do not invent or assume information.
4. If the answer cannot be found in the context,
   say exactly:

   "I could not find the answer in the provided PDF."

5. Give a clear and concise answer.
6. If possible, mention relevant details from the document.

--------------------------------------------------

PDF CONTEXT:

{context}

--------------------------------------------------

USER QUESTION:

{question}

--------------------------------------------------

ANSWER:
"""


    try:

        response = (
            client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )
        )


        # Safely extract response text
        answer = getattr(
            response,
            "text",
            None
        )


        if not answer:

            answer = (
                "The model returned an empty response."
            )


        return (
            answer,
            relevant_documents
        )


    except Exception as e:

        error_message = f"""
Error while generating answer.

Model used:
{MODEL_NAME}

Error:
{str(e)}

Try changing the MODEL_NAME variable to one of
the models displayed earlier in the available
models list.
"""

        return (
            error_message,
            relevant_documents
        )


print(
    "Question-answering chatbot created successfully!"
)


# 16. DISPLAY SOURCE DOCUMENTS
# ------------------------------------------------------------

print("\n")
print("=" * 70)

print(
    "SOURCE DOCUMENTS USED"
)

print("=" * 70)


for i, doc in enumerate(
    sources
):

    page_number = (
        doc.metadata.get(
            "page",
            "Unknown"
        )
    )


    if isinstance(
        page_number,
        int
    ):
        page_number += 1


    print(
        f"\nSOURCE {i + 1}"
    )

    print(
        "PDF Page:",
        page_number
    )

    print(
        "\nContent Preview:"
    )

    print(
        doc.page_content[:500]
    )

    print(
        "\n" + "-" * 60
    )


# ------------------------------------------------------------
# 18. START INTERACTIVE CHATBOT
# ------------------------------------------------------------

print("\n")
print("=" * 70)

print(
    "PDF QUESTION-ANSWERING CHATBOT"
)

print("=" * 70)

print(
    "\nAsk questions about your PDF."
)

print(
    "Type 'exit' or 'quit' to stop."
)


while True:

    user_question = input(
        "\nYou: "
    )


    # Stop chatbot
    if user_question.lower().strip() in [
        "exit",
        "quit"
    ]:

        print(
            "\nChatbot: Goodbye!"
        )

        break


    # Ignore empty questions
    if not user_question.strip():

        print(
            "\nChatbot: Please enter a question."
        )

        continue


    # Generate answer
    answer, sources = ask_pdf(
        user_question
    )


    print(
        "\nChatbot:"
    )

    print(
        answer
    )


    # Display pages used
    print(
        "\nSources used:"
    )


    if sources:

        for i, doc in enumerate(
            sources
        ):

            page_number = (
                doc.metadata.get(
                    "page",
                    "Unknown"
                )
            )


            if isinstance(
                page_number,
                int
            ):
                page_number += 1


            print(
                f"Source {i + 1}: "
                f"PDF Page {page_number}"
            )


    else:

        print(
            "No source documents found."
        )


    print(
        "\n" + "-" * 70
    )

All libraries imported successfully!
Gemini API key configured successfully!
Gemini client initialized successfully!

Checking available Gemini models...

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/g

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!

Creating FAISS vector database...
FAISS vector database created successfully!
Retriever function created successfully!
Question-answering chatbot created successfully!


SOURCE DOCUMENTS USED

SOURCE 1
PDF Page: 2

Content Preview:
Sd/..
Controller of Examination
EVEN-JUNIOR SH7801-Human Values and Professional Ethics 9 A+ 2 Pass
Credit Registered : 28 Credit Completed : 28
Semester Course Name Grade point Grade Credit Result status

------------------------------------------------------------

SOURCE 2
PDF Page: 1

Content Preview:
Saveetha Engineering College (Autonomous)
Saveetha Nagar ,Thandalam ,Chennai
End Semester Examination - Jun-2026
Student Name: MANORAJAPRIYAN L E Reg. No.: 212225040227
Semester: EVEN-JUNIOR Gender: Male
Program: B.E. Computer Science and Engineering
EVEN-JUNIOR SH3214-Physics for Quantum Computing 10 S 3 Pass
EVEN-JUNIOR CS3414-Fundamentals of Web Application Development 9 A+ 5 Pass
EVEN-JUNIOR CS3304-Fundamentals of C


You:  What is the overall result mentioned in this document?



Chatbot:
I could not find the answer in the provided PDF.

Sources used:
Source 1: PDF Page 2
Source 2: PDF Page 1

----------------------------------------------------------------------
